# ADMM MPC

This notebook runs a deployable rolling ADMM-MPC controller with a transformer export surrogate.

Unlike the standalone `ADMM.ipynb` oracle notebook, this notebook only uses the current rolling forecast window, executes the first control action, and then re-solves at the next real timestep. It is also different from the existing single-agent `MPC + LSTM` workflow because the optimization is distributed across agents and coupled through the transformer export surrogate.


In [ ]:
from pathlib import Path
import importlib
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

repo_root = Path.cwd().resolve()
while repo_root != repo_root.parent and not (repo_root / "configs").exists():
    repo_root = repo_root.parent
if not (repo_root / "configs").exists():
    raise RuntimeError("Could not locate the project root from the notebook working directory.")
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import configs as configs_pkg
from configs.profiles import compose_experiment_config
from scripts.mainline_compare import compare_rollout_metrics
from scripts.utils import admm_mpc_notebook_helpers as admm_mpc_nb
from scripts.utils import grid_notebook_workflow as grid_nb
from scripts.utils.project_paths import project_root as resolve_project_root

configs_pkg = importlib.reload(configs_pkg)
admm_mpc_nb = importlib.reload(admm_mpc_nb)
grid_nb = importlib.reload(grid_nb)


In [ ]:
PROJECT_ROOT = resolve_project_root()
DATA_DIR = PROJECT_ROOT / "data"

TEST_START_DATE = "2020-06-01"
TEST_END_DATE = "2020-06-30"
PREDICTION_MODE = "normal"
W_SOC_PEN = 2.0
LOAD_SCALE = [10.0] * 5
PV_SCALE = [5.0] * 5
BATTERY_CAPACITY_KWH = 20.0
BATTERY_MAX_POWER_KW = 10.0
BATTERY_MAX_CHARGE_RATE = BATTERY_MAX_POWER_KW / BATTERY_CAPACITY_KWH
BATTERY_CONTROLS = {"battery_capacity": BATTERY_CAPACITY_KWH, "max_charge_rate": BATTERY_MAX_CHARGE_RATE}
SHOW_PROGRESS = True
RHO_INIT = None
RHO_MIN = 1e-3
RHO_MAX = 1e3
RHO_ADAPTATION = "residual_balancing"
MAX_ITERS = 100
MAX_ITERS_FIRST_STEP = 300
PRIMAL_TOL = 1e-3
DUAL_TOL = 1e-3
TERMINAL_COST_MULTIPLIER = 1.0


In [ ]:
cfg = compose_experiment_config(
    profile="base",
    algorithm="MATD3",
    model_family="mlp",
    data_dir=DATA_DIR,
    runtime_mode="performance",
)
grid_nb.apply_notebook_experiment_settings(
    cfg,
    prediction_mode=PREDICTION_MODE,
    test_start_date=TEST_START_DATE,
    test_end_date=TEST_END_DATE,
    load_scale=LOAD_SCALE,
    pv_scale=PV_SCALE,
    battery_controls=BATTERY_CONTROLS,
)
cfg.reward.w_soc_pen = W_SOC_PEN
cfg.env.episode_limit = int(round(24.0 / cfg.env.dt))
if abs(float(cfg.env.dt) * int(cfg.env.episode_limit) - 24.0) > 1e-9:
    raise ValueError("ADMM_mpc notebook expects one full day per episode.")

display(
    pd.Series(
        {
            "test_start_date": cfg.data.test_start_date,
            "test_end_date": cfg.data.test_end_date,
            "prediction_mode": PREDICTION_MODE,
            "forecast_backend": cfg.forecast.type,
            "import_price_markup_eur_per_kwh": float(cfg.reward.import_price_markup_eur_per_kwh),
            "export_subsidy_eur_per_kwh": float(cfg.reward.export_subsidy_eur_per_kwh),
            "future_horizon": cfg.env.future_horizon,
            "episode_limit": cfg.env.episode_limit,
            "n_days": int((pd.Timestamp(TEST_END_DATE) - pd.Timestamp(TEST_START_DATE)).days + 1),
        },
        name="admm_mpc_notebook_config",
    )
)


In [ ]:
admm_mpc_rollout = admm_mpc_nb.collect_admm_mpc_rollout(
    cfg,
    prediction_mode=PREDICTION_MODE,
    show_progress=SHOW_PROGRESS,
    rho_init=RHO_INIT,
    rho_min=RHO_MIN,
    rho_max=RHO_MAX,
    rho_adaptation=RHO_ADAPTATION,
    max_iters=MAX_ITERS,
    max_iters_first_step=MAX_ITERS_FIRST_STEP,
    primal_tol=PRIMAL_TOL,
    dual_tol=DUAL_TOL,
    terminal_cost_multiplier=TERMINAL_COST_MULTIPLIER,
)


In [ ]:
metrics_df = compare_rollout_metrics(admm_mpc_rollout)
display(metrics_df)

diagnostic_summary = pd.Series(
    {
        "controller": admm_mpc_rollout.meta["controller"],
        "forecast_backend": admm_mpc_rollout.meta["forecast_backend"],
        "convergence_rate": float(admm_mpc_rollout.step_df["admm_converged"].mean()),
        "avg_admm_iterations": float(admm_mpc_rollout.step_df["admm_iterations"].mean()),
        "max_primal_residual": float(admm_mpc_rollout.step_df["admm_final_primal_residual"].max()),
        "max_dual_residual": float(admm_mpc_rollout.step_df["admm_final_dual_residual"].max()),
        "avg_solve_time_sec": float(admm_mpc_rollout.step_df["admm_solve_time_sec"].mean()),
    },
    name="admm_mpc_diagnostics",
)
display(diagnostic_summary)

display(
    pd.Series(
        {
            "controller": admm_mpc_rollout.meta["controller"],
            "forecast_backend": admm_mpc_rollout.meta["forecast_backend"],
            "admm_terminal_cost_mode": admm_mpc_rollout.meta.get("admm_terminal_cost_mode"),
        },
        name="admm_mpc_rollout_summary",
    )
)


In [ ]:
grid_nb.plot_rollout_dashboard(admm_mpc_rollout)
plt.show()

grid_nb.plot_voltage_profile_comparison(admm_mpc_rollout)
plt.show()

grid_nb.plot_power_balance_comparison(admm_mpc_rollout)
plt.show()

grid_nb.plot_net_load_comparison(admm_mpc_rollout)
plt.show()

grid_nb.plot_battery_power_and_soc_comparison(admm_mpc_rollout)
plt.show()
